# 07_classical_model_development_and_cv

Develop and lock the classical baseline model using Notebook 06 outputs. The independent test set is not used for model selection or threshold selection.

## Methodological safeguards
- Uses only expert-assigned clinical labels stored in the curated folder-derived feature table from Notebook 06.
- Verifies horse-level split exclusivity before modelling.
- Excludes hotspot/annotation metadata from predictors.
- Selects model and decision threshold on training/validation data only.

In [1]:

from pathlib import Path
import os, json, shutil, zipfile, hashlib, warnings, math, random, glob
from datetime import datetime, timezone
import pandas as pd
import numpy as np

BASE_DIR = Path(os.environ.get("THERMO_BASE_DIR", "/content"))
PROJECT_NAME = "project_thermography_equine"
PROJECT_ROOT = BASE_DIR / PROJECT_NAME
DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
METADATA_DIR = DATA_ROOT / "metadata"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
MODEL_SELECTION_DIR = OUTPUT_ROOT / "model_selection"
PROCESSED_DIR = DATA_ROOT / "processed"
CLEAN_IMAGE_DIR = PROCESSED_DIR / "clean_images"
FEATURES_DIR = OUTPUT_ROOT / "features"

for p in [PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, SPLIT_DATA_DIR, METADATA_DIR, ANNOTATIONS_DIR,
          OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR,
          MODEL_SELECTION_DIR, PROCESSED_DIR, CLEAN_IMAGE_DIR, FEATURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SEARCH_ROOTS = [BASE_DIR, Path('/mnt/data')]

def _existing_roots():
    return [p for p in SEARCH_ROOTS if p.exists()]

def _safe_copy(src: Path, dst: Path, overwrite: bool = False):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.resolve() == dst.resolve():
        return dst
    if overwrite or not dst.exists():
        shutil.copy2(src, dst)
    return dst

def find_first(patterns, roots=None, exclude_dirs=(".ipynb_checkpoints",)):
    roots = roots or _existing_roots()
    for root in roots:
        for pattern in patterns:
            for p in sorted(root.rglob(pattern)):
                if any(part in exclude_dirs for part in p.parts):
                    continue
                if p.is_file():
                    return p
    return None

def recover_notebook06_outputs():
    """Recover outputs from Notebook 06 if the user uploaded them to /content instead of project outputs/config."""
    targets = {
        "classical_features.csv": ["classical_features.csv", "classical_features*.csv"],
        "classical_feature_metadata.json": ["classical_feature_metadata.json"],
        "classical_feature_summary.csv": ["classical_feature_summary.csv"],
    }
    recovered = {}
    for target, patterns in targets.items():
        dst = CONFIG_DIR / target
        if dst.exists():
            recovered[target] = str(dst)
            continue
        src = find_first(patterns)
        if src is None:
            recovered[target] = None
            continue
        _safe_copy(src, dst)
        if target.endswith('.csv'):
            _safe_copy(src, FEATURES_DIR / target)
        recovered[target] = str(src)
    meta_path = CONFIG_DIR / "classical_feature_metadata.json"
    if meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))

        if meta.get("clinical_label_source") == "folder_structure":
            meta["clinical_label_source"] = "expert_curated_folder_structure_based_on_clinical_assessment"
            meta["clinical_label_source_note"] = (
                "Folder structure stores labels assigned by veterinary specialists before model development; "
                "folder names are a technical representation, not an independent diagnostic criterion."
            )
            meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return recovered

def load_classical_features_and_metadata():
    recovered = recover_notebook06_outputs()
    features_path = CONFIG_DIR / "classical_features.csv"
    metadata_path = CONFIG_DIR / "classical_feature_metadata.json"
    if not features_path.exists() or not metadata_path.exists():
        raise FileNotFoundError(
            "Notebook 06 outputs are missing. Upload classical_features.csv and classical_feature_metadata.json, "
            "or run Notebook 06 before this notebook. Recovery status: " + json.dumps(recovered, indent=2)
        )
    features = pd.read_csv(features_path)
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    feature_columns = metadata.get("feature_columns", [])
    required = {"horse_id", "split", "label_clinical", "label_binary"}
    missing_required = sorted(required - set(features.columns))
    if missing_required:
        raise KeyError("classical_features.csv is missing required columns: " + ", ".join(missing_required))
    missing_features = [c for c in feature_columns if c not in features.columns]
    if missing_features:
        raise KeyError("Feature columns listed in metadata are absent from classical_features.csv: " + ", ".join(missing_features[:20]))
    allowed_splits = {"train", "valid", "test"}
    bad_splits = sorted(set(features["split"].dropna().astype(str)) - allowed_splits)
    if bad_splits:
        raise ValueError("Unexpected split labels: " + ", ".join(bad_splits))
    overlap = []
    for a in allowed_splits:
        for b in allowed_splits:
            if a < b:
                ia = set(features.loc[features["split"] == a, "horse_id"].astype(str))
                ib = set(features.loc[features["split"] == b, "horse_id"].astype(str))
                common = ia & ib
                if common:
                    overlap.append((a, b, len(common), sorted(list(common))[:5]))
    if overlap:
        raise ValueError("Horse-level leakage detected across splits: " + repr(overlap))
    leakage_like = [c for c in feature_columns if any(token in c.lower() for token in ["hotspot", "annotation", "label", "split", "horse_id", "image_name"])]
    if leakage_like:
        raise ValueError("Potential leakage columns found among predictors: " + ", ".join(leakage_like))
    print("Project root:", PROJECT_ROOT)
    print("Config dir:", CONFIG_DIR)
    print("Loaded Notebook 06 feature table:", features.shape)
    print("Clinical label source:", metadata.get("clinical_label_source"))
    display(features.groupby(["split", "label_clinical"]).size().reset_index(name="n"))
    return features, metadata, feature_columns


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import joblib

features, feature_metadata, feature_columns = load_classical_features_and_metadata()


Project root: /content/project_thermography_equine
Config dir: /content/project_thermography_equine/outputs/config
Loaded Notebook 06 feature table: (347, 46)
Clinical label source: expert_classification_from_folder_structure


,split,label_clinical,n
0,test,healthy,40
1,test,pathological,13
2,train,healthy,179
3,train,pathological,63
4,valid,healthy,38
5,valid,pathological,14


In [3]:
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

def get_prob(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    scores = model.decision_function(X)
    return 1 / (1 + np.exp(-scores))

def choose_threshold_youden(y_true, y_prob):
    # Validation-only threshold choice: maximizes sensitivity + specificity - 1.
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    candidates = np.unique(np.r_[0.0, y_prob, 1.0])
    best = {"threshold": 0.5, "youden_j": -np.inf, "balanced_accuracy": -np.inf}
    for thr in candidates:
        pred = (y_prob >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        sens = tp / (tp + fn) if (tp + fn) else 0.0
        spec = tn / (tn + fp) if (tn + fp) else 0.0
        youden = sens + spec - 1
        bal = (sens + spec) / 2
        if (youden, bal) > (best["youden_j"], best["balanced_accuracy"]):
            best = {"threshold": float(thr), "youden_j": float(youden), "balanced_accuracy": float(bal), "sensitivity": float(sens), "specificity": float(spec)}
    return best

def metric_dict(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan,
        "average_precision": float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "sensitivity": float(tp / (tp + fn)) if (tp + fn) else np.nan,
        "specificity": float(tn / (tn + fp)) if (tn + fp) else np.nan,
        "ppv": float(tp / (tp + fp)) if (tp + fp) else np.nan,
        "npv": float(tn / (tn + fn)) if (tn + fn) else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


In [4]:
train_df = features[features["split"] == "train"].copy()
valid_df = features[features["split"] == "valid"].copy()
if train_df.empty or valid_df.empty:
    raise ValueError("Train and validation sets are required for Notebook 07.")

X_train = train_df[feature_columns]
y_train = train_df["label_binary"].astype(int).values
X_valid = valid_df[feature_columns]
y_valid = valid_df["label_binary"].astype(int).values

models = {
    "logistic_l2": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_SEED)),
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=250, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=1, min_samples_leaf=2)),
    ]),
    "gradient_boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(random_state=RANDOM_SEED)),
    ]),
}

# SVM with probability calibration can be slow; enable explicitly if needed for exploratory comparison.
if os.environ.get("THERMO_INCLUDE_SVM", "0") == "1":
    from sklearn.svm import SVC
    models["svm_rbf"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_SEED)),
    ])

min_class_count = int(np.min(np.bincount(y_train)))
n_splits = max(2, min(5, min_class_count))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

rows, fitted, valid_predictions = [], {}, {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)
    model.fit(X_train, y_train)
    fitted[name] = model
    valid_prob = get_prob(model, X_valid)
    valid_predictions[name] = valid_prob
    threshold_info = choose_threshold_youden(y_valid, valid_prob)
    md = metric_dict(y_valid, valid_prob, threshold=threshold_info["threshold"])
    rows.append({
        "model_name": name,
        "train_cv_auc_mean": float(np.mean(cv_scores)),
        "train_cv_auc_sd": float(np.std(cv_scores, ddof=1)) if len(cv_scores) > 1 else np.nan,
        "n_cv_splits": int(n_splits),
        "valid_selected_threshold": threshold_info["threshold"],
        "valid_youden_j": threshold_info["youden_j"],
        **{f"valid_{k}": v for k, v in md.items()},
    })

model_selection = pd.DataFrame(rows).sort_values(
    ["valid_roc_auc", "valid_balanced_accuracy", "train_cv_auc_mean"],
    ascending=False,
).reset_index(drop=True)
display(model_selection)
selected_model_name = model_selection.loc[0, "model_name"]
selected_threshold = float(model_selection.loc[0, "valid_selected_threshold"])
selected_model = fitted[selected_model_name]
print("Selected classical model:", selected_model_name)
print("Validation-selected threshold:", selected_threshold)


,model_name,train_cv_auc_mean,train_cv_auc_sd,n_cv_splits,valid_selected_threshold,valid_youden_j,valid_roc_auc,valid_average_precision,valid_accuracy,valid_balanced_accuracy,valid_f1,valid_sensitivity,valid_specificity,valid_ppv,valid_npv,valid_tn,valid_fp,valid_fn,valid_tp
0,logistic_l2,0.772385,0.052791,5,0.212146,0.394737,0.697368,0.403760,0.557692,0.697368,0.549020,1.000000,0.394737,0.378378,1.000000,15,23,0,14
1,random_forest,0.813941,0.058689,5,0.104888,0.263158,0.639098,0.400932,0.461538,0.631579,0.500000,1.000000,0.263158,0.333333,1.000000,10,28,0,14
2,gradient_boosting,0.807995,0.073290,5,0.446448,0.225564,0.582707,0.366900,0.730769,0.612782,0.416667,0.357143,0.868421,0.500000,0.785714,33,5,9,5


Selected classical model: logistic_l2
Validation-selected threshold: 0.21214616001573788


In [5]:
# Refit selected model on train + validation after model and threshold have been selected.
dev_df = features[features["split"].isin(["train", "valid"])].copy()
X_dev = dev_df[feature_columns]
y_dev = dev_df["label_binary"].astype(int).values
selected_model.fit(X_dev, y_dev)

joblib.dump(selected_model, MODELS_DIR / "selected_classical_model.joblib")
model_selection.to_csv(MODEL_SELECTION_DIR / "classical_model_selection_results.csv", index=False)
model_selection.to_csv(REPORTS_DIR / "classical_model_selection_results.csv", index=False)
model_selection.to_csv(TABLES_DIR / "table_classical_model_selection_results.csv", index=False)

selection_record = {
    "selected_model_name": selected_model_name,
    "selection_metric": "validation ROC AUC, tie-broken by validation balanced accuracy and training CV AUC",
    "decision_threshold": selected_threshold,
    "threshold_selection_method": "Youden index on the validation set only",
    "test_set_used_for_selection": False,
    "n_train": int(len(train_df)),
    "n_valid": int(len(valid_df)),
    "n_development": int(len(dev_df)),
    "feature_columns": feature_columns,
    "clinical_label_source": feature_metadata.get("clinical_label_source"),
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(CONFIG_DIR / "selected_classical_model.json").write_text(json.dumps(selection_record, indent=2), encoding="utf-8")
(REPORTS_DIR / "selected_classical_model.json").write_text(json.dumps(selection_record, indent=2), encoding="utf-8")
print(json.dumps(selection_record, indent=2)[:2000])


{
  "selected_model_name": "logistic_l2",
  "selection_metric": "validation ROC AUC, tie-broken by validation balanced accuracy and training CV AUC",
  "decision_threshold": 0.21214616001573788,
  "threshold_selection_method": "Youden index on the validation set only",
  "test_set_used_for_selection": false,
  "n_train": 242,
  "n_valid": 52,
  "n_development": 294,
  "feature_columns": [
    "img_mean",
    "img_std",
    "img_min",
    "img_max",
    "img_p05",
    "img_p10",
    "img_p50",
    "img_p90",
    "img_p95",
    "img_p99",
    "top10_mean",
    "top10_std",
    "top10_area_fraction",
    "top30_mean",
    "top30_std",
    "top30_area_fraction",
    "entropy_gray",
    "lr_asym_mean_abs",
    "lr_asym_max_abs",
    "lr_asym_std",
    "height_px_features",
    "width_px_features",
    "r_mean",
    "r_std",
    "r_p90",
    "g_mean",
    "g_std",
    "g_p90",
    "b_mean",
    "b_std",
    "b_p90",
    "grad_mean",
    "grad_std",
    "grad_p90",
    "grad_p95"
  ],
  "clin